In [1]:
# Load the extension
%load_ext autoreload
# Set to reload all modules (except those explicitly excluded) before executing user code
%autoreload 2

In [2]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [3]:
from tqdm import tqdm
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
def check_policy(Q):
    result_list = []
    for _dist in range(5):
        for _dir in range(4):
            result = np.argmax(Q.get(((_dir,), _dist, None, None), [0])) == _dir
            result_list.append(result)
    return np.mean(result_list)

In [7]:
from src.envs.kutulu_world import KutuluWorldEnv, DEFAULT_KUTULU_ACTIONS
from src.envs.distance import find_path
from src.envs.kutulu_player import KutuluPlayer
from src.envs.strategy import RandomStrategy, GreedyStrategy, LazyGreedyStrategy

In [8]:
env = KutuluWorldEnv(
    server_host='localhost:8080',
    maze_name="Hypersonic",
    league_level=1,
    players_count=4,
)

In [9]:
player = KutuluPlayer(env)

In [18]:
pi = LazyGreedyStrategy()
eps=0.05
alpha=0.1
gamma=0.99
num_experiments=10000
can_wait=True
verbose=False

In [19]:
Q = pi.Q
reward_list = []
quality_list = []

In [20]:
for i in tqdm(range(num_experiments)):
    observation, info = env.reset()
    rollout_rewards = []
    game_over = False
    while not game_over:
        state = player.get_state(0, max_explorer_dist=5, max_wanderer_dist=5)
        player_mask = ~np.array(player.env.get_valid_action_mask()[0])
        At = pi.getActionEpsGreedyMasked(state, len(DEFAULT_KUTULU_ACTIONS), eps, player_mask)
        action = env.sample_valid_action(can_wait)
        action[0] = At
        entities, rewards, game_over, info = env.step(action)
        reward = rewards[0]
        
        if reward is None:
            break
        
        if game_over:
            Q[state][At] += alpha * (reward - Q[state][At])
            break

        Q[state][At] += alpha * (
            reward + gamma * Q[state].max() - Q[state][At]
        )
        
        if verbose:
            viz_map(env.map, entities['entities'])

        rollout_rewards.append(reward)
    reward_list.append(sum(rollout_rewards))
    quality_list.append(check_policy(pi.Q))
    if i % 10 == 0:
        print(f"i={i}\tmean_reward={np.mean(reward_list):.2f}\tquality={quality_list[-1]:.2f}")

  0%|          | 2/10000 [00:00<32:22,  5.15it/s]

i=0	mean_reward=-196.00	quality=0.25


  0%|          | 11/10000 [00:02<38:21,  4.34it/s]

i=10	mean_reward=-177.45	quality=0.20


  0%|          | 22/10000 [00:04<36:53,  4.51it/s]

...

i=260	mean_reward=-133.92	quality=0.25


  3%|▎         | 271/10000 [01:50<1:17:45,  2.09it/s]

i=270	mean_reward=-133.62	quality=0.25


  3%|▎         | 279/10000 [01:54<1:06:44,  2.43it/s]


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(np.cumsum(reward_list) / (np.arange(len(reward_list)) + 1))
_ = plt.grid()

In [13]:
import pickle as pkl
import zlib
import base64

In [14]:
len([k for k in pi.Q.keys()])

538

In [15]:
new_keys = [k for k in pi.Q.keys() if ((k[1] or 0) < 8) and ((k[3] or 0) < 5)]

In [16]:
len(new_keys)

410

In [17]:
data1 = pkl.dumps(np.array(list(pi.Q[k] for k in new_keys)))
data2 = pkl.dumps(list(new_keys))

In [191]:
# base64.b64encode(zlib.compress(data2, level=9))